# Agent 2 local production execution

This is an executed adaptation of `Agent2_University_Agent.ipynb`, generated by
`scripts/run_agent2_notebook.py`. Original uploads remain unchanged.

Local adaptations:
- Read repository CSVs and export all fourteen artifacts into one new directory.
- Reuse the notebook's selected five features, k=2 applicant clusters, k=10
  program clusters, and recorded GradientBoosting settings (300 trees,
  depth 3, learning rate 0.1). Historical model/feature searches are not repeated.
- Use disjoint fit/validation/test rows (64%/16%/20%); fit applicant preprocessing,
  the classifier and calibration only on fit rows. The original notebook
  incorrectly included its threshold-validation rows in model fitting.
- Compute applicant silhouette on a fixed sample of 3000 rows to bound memory;
  this diagnostic does not choose k in this production rerun.
- Run SHAP and export cells; replace Colab upload/download with local paths.
- Exercise the application worker separately with `tests/test_agent2_live.py`. Cloud retrieval, Groq,
  Postgres and the inconsistent historical final demo are not executed.

The dataset's identity/provenance issues remain unresolved. Row-holdout metrics
describe this local dataset only; they do not establish unseen-student or real
admissions accuracy. This run is for a functional frontend integration.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, importlib.metadata
ROOT = Path('C:\\Users\\ASUS\\Downloads\\Meridian\\Major_Project-2027-main')
EXPORT_DIR = Path('C:\\Users\\ASUS\\Downloads\\Meridian\\Major_Project-2027-main\\backend\\app\\agents\\university_matching_agent\\trained_models_local')
REPORT_DIR = Path('C:\\Users\\ASUS\\Downloads\\Meridian\\Major_Project-2027-main\\reports\\agent2-local')
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Python:', sys.version)
print('Export directory:', EXPORT_DIR)


Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
Export directory: C:\Users\ASUS\Downloads\Meridian\Major_Project-2027-main\backend\app\agents\university_matching_agent\trained_models_local


In [2]:
import numpy as np
import pandas as pd

train = pd.read_csv(ROOT / "data/raw/agent2/agent2_training.csv", low_memory=False)
catalog = pd.read_csv(ROOT / "data/raw/agent2/agent2_enhanced_program_catalog.csv", low_memory=False)

print("=== agent2_training.csv (replaces us_historical_admits_no_phd.csv this round) ===")
print("shape:", train.shape)
print("duplicate rows:", train.duplicated().sum())
print("duplicate applicant_id (same applicant, multiple programs applied to):", train["applicant_id"].duplicated().sum())
print("unique program_id:", train["program_id"].nunique())
print("target distribution (admit_result):\n", train["admit_result"].value_counts(normalize=True))
print("target_admission_status breakdown:\n", train["target_admission_status"].value_counts(normalize=True))

print("\n=== agent2_enhanced_program_catalog.csv (used here only for university-level clustering columns) ===")
print("shape:", catalog.shape)

match_rate = train["program_id"].isin(catalog["program_id"]).mean()
print(f"\nagent2_training.csv program_id coverage in catalog: {match_rate:.1%}")

# agent2_training.csv already carries min_gpa/min_gre_quant/min_gre_verbal/program_admit_rate/tier_score
# per row (pre-merged with a catalog snapshot). Only pull in the extra university-level columns this
# pipeline needs for clustering (Section 6e) that are NOT already in agent2_training.csv.
EXTRA_CATALOG_COLS = ["university_admission_rate", "us_news_ranking", "world_ranking",
                       "tuition_out_state_usd", "student_size", "stem_designated"]
m = train.merge(catalog[["program_id"] + EXTRA_CATALOG_COLS], on="program_id", how="left")
print(f"merged shape: {m.shape}")
for c in EXTRA_CATALOG_COLS:
    if m[c].dtype != bool:
        m[c] = pd.to_numeric(m[c], errors="coerce")
        m[c] = m[c].fillna(m[c].median())

m["gpa_numeric"] = m["gpa"]
m["gre_numeric"] = m["gre_quant"]
m["gre_missing"] = m["gre_numeric"].isna().astype(int)
m["y_binary"] = m["admit_result"].astype(int)

# ============================================================
# 6b. Leakage audit (Part 10)
# ============================================================


=== agent2_training.csv (replaces us_historical_admits_no_phd.csv this round) ===
shape: (15988, 26)
duplicate rows: 0
duplicate applicant_id (same applicant, multiple programs applied to): 3600
unique program_id: 3033
target distribution (admit_result):
 admit_result
0    0.662247
1    0.337753
Name: proportion, dtype: float64
target_admission_status breakdown:
 target_admission_status
Deny        0.649737
Admit       0.337753
Waitlist    0.012509
Name: proportion, dtype: float64

=== agent2_enhanced_program_catalog.csv (used here only for university-level clustering columns) ===
shape: (3052, 24)

agent2_training.csv program_id coverage in catalog: 100.0%
merged shape: (15988, 32)


In [3]:
# Fixed production choices recorded by the supplied notebook and feature export.
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
FINAL_FEATURES = joblib.load(ROOT / 'backend/app/agents/university_matching_agent/trained models/agent2_feature_columns.joblib')
assert FINAL_FEATURES == ['tier_score', 'gpa_numeric', 'gre_numeric', 'gre_missing', 'applicant_cluster']
AGENT1_TIER_FEATURES = ['tier_score']
PRODUCTION_MODEL_NAME = 'GradientBoosting'
PRODUCTION_PARAMS = dict(n_estimators=300, max_depth=3, learning_rate=0.1, random_state=42)
assert m[['gpa_numeric','tier_score','y_binary']].notna().all().all()
print('Fixed features:', FINAL_FEATURES)
print('Fixed classifier settings:', PRODUCTION_PARAMS)


Fixed features: ['tier_score', 'gpa_numeric', 'gre_numeric', 'gre_missing', 'applicant_cluster']
Fixed classifier settings: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'random_state': 42}


In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

y = m["y_binary"]
pool_idx, test_idx = train_test_split(m.index, test_size=0.2, random_state=42, stratify=y)
fit_idx, val_idx = train_test_split(pool_idx, test_size=0.2, random_state=42, stratify=y.loc[pool_idx])

# --- Applicant clustering (2 features only -- see limitation above) ---
APPLICANT_CLUSTER_FEATURES = ["gpa_numeric", "gre_numeric"]
applicant_cluster_medians = m.loc[fit_idx, APPLICANT_CLUSTER_FEATURES].median().to_dict()
m[APPLICANT_CLUSTER_FEATURES] = m[APPLICANT_CLUSTER_FEATURES].fillna(pd.Series(applicant_cluster_medians))

applicant_scaler = StandardScaler().fit(m.loc[fit_idx, APPLICANT_CLUSTER_FEATURES])
applicant_scaled = applicant_scaler.transform(m.loc[fit_idx, APPLICANT_CLUSTER_FEATURES])

best_k_app, best_sil_app = None, -1
for k in [2]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(applicant_scaled)
    sil = silhouette_score(applicant_scaled, labels, sample_size=3000, random_state=42)
    db = davies_bouldin_score(applicant_scaled, labels)
    print(f"applicant k={k}: silhouette={sil:.3f}  davies-bouldin={db:.3f}")
    if sil > best_sil_app:
        best_k_app, best_sil_app = k, sil
print(f"Selected k={best_k_app} (silhouette={best_sil_app:.3f} -- inflated by having only 2 input "
      f"dimensions; not a strong clustering in the sense the brief is asking about)")

applicant_kmeans = KMeans(n_clusters=best_k_app, random_state=42, n_init=10).fit(applicant_scaled)
m.loc[fit_idx, "applicant_cluster"] = applicant_kmeans.labels_
m.loc[m.index.difference(fit_idx), "applicant_cluster"] = applicant_kmeans.predict(
    applicant_scaler.transform(m.loc[m.index.difference(fit_idx), APPLICANT_CLUSTER_FEATURES])
)
m["applicant_cluster"] = m["applicant_cluster"].astype(int)

# --- University/program clustering (richer feature set) ---
UNIV_CLUSTER_COLS = ["us_news_ranking", "world_ranking", "tuition_out_state_usd",
                      "program_admit_rate", "university_admission_rate", "tier_score",
                      "min_gpa", "min_gre_quant", "student_size"]
programs_clean = catalog[["program_id"] + UNIV_CLUSTER_COLS].drop_duplicates("program_id")
univ_input_raw = programs_clean[UNIV_CLUSTER_COLS].apply(pd.to_numeric, errors="coerce")
univ_cluster_medians = univ_input_raw.median().to_dict()
univ_input = univ_input_raw.fillna(pd.Series(univ_cluster_medians))
university_cluster_scaler = StandardScaler().fit(univ_input)
univ_scaled = university_cluster_scaler.transform(univ_input)

best_k_univ, best_sil_univ = None, -1
for k in [10]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(univ_scaled)
    sil = silhouette_score(univ_scaled, labels)
    if sil > best_sil_univ:
        best_k_univ, best_sil_univ = k, sil
print(f"\nSelected k={best_k_univ} program clusters (silhouette={best_sil_univ:.3f})")

university_cluster_model = KMeans(n_clusters=best_k_univ, random_state=42, n_init=10).fit(univ_scaled)
programs_clean = programs_clean.copy()
programs_clean["program_cluster"] = university_cluster_model.labels_
m = m.drop(columns=["program_cluster"], errors="ignore").merge(
    programs_clean[["program_id", "program_cluster"]], on="program_id", how="left"
)

print(f"\nApplicant clustering artifacts saved for reuse: scaler + KMeans(k={best_k_app}), "
      f"features={APPLICANT_CLUSTER_FEATURES}, medians={applicant_cluster_medians}")
print(f"University/program clustering artifacts saved for reuse: scaler + KMeans(k={best_k_univ}), "
      f"features={UNIV_CLUSTER_COLS}, medians used for missing inputs={univ_cluster_medians}")


applicant k=2: silhouette=0.417  davies-bouldin=0.895
Selected k=2 (silhouette=0.417 -- inflated by having only 2 input dimensions; not a strong clustering in the sense the brief is asking about)



Selected k=10 program clusters (silhouette=0.403)

Applicant clustering artifacts saved for reuse: scaler + KMeans(k=2), features=['gpa_numeric', 'gre_numeric'], medians={'gpa_numeric': 3.273224664441744, 'gre_numeric': 160.66666666666666}
University/program clustering artifacts saved for reuse: scaler + KMeans(k=10), features=['us_news_ranking', 'world_ranking', 'tuition_out_state_usd', 'program_admit_rate', 'university_admission_rate', 'tier_score', 'min_gpa', 'min_gre_quant', 'student_size'], medians used for missing inputs={'us_news_ranking': 32.0, 'world_ranking': 73.0, 'tuition_out_state_usd': 49922.0, 'program_admit_rate': 0.4267738883632922, 'university_admission_rate': 0.4008, 'tier_score': 74.23, 'min_gpa': 3.364301482182277, 'min_gre_quant': 156.38442131819616, 'student_size': 20443.0}


In [5]:
# --- Agent 1 Tier: consistency check + the exact encoding used at train AND inference time ---
assert "tier_score" in FINAL_FEATURES, (
    "Agent 1 Tier (tier_score) must be part of the production feature set -- "
    "it is not optional and must never be removed by feature selection."
)

# tier_score is used as-is: a continuous ~0-100 competitiveness score that is Agent 1's
# output, already joined into every row. No re-binning/re-encoding happens anywhere, so
# train-time and inference-time representations are identical by construction. This
# constant documents that representation so inference code (Section 8) can assert
# against it instead of silently assuming it.
AGENT1_TIER_REPRESENTATION = {
    "column": "tier_score",
    "dtype": "float",
    "valid_range": (float(m["tier_score"].min()), float(m["tier_score"].max())),
    "source": "agent2_training.csv (pre-merged with agent2_enhanced_program_catalog.csv)",
}

invalid_tier_rows = int(m["tier_score"].isna().sum())
if invalid_tier_rows:
    raise ValueError(
        f"{invalid_tier_rows} rows have a missing/invalid Agent 1 tier_score after the merge. "
        "Agent 2 will not silently default these -- fix the upstream join before training."
    )
print("Agent 1 Tier representation locked in:", AGENT1_TIER_REPRESENTATION)

# Applicant/university clustering features, if selected into FINAL_FEATURES, reuse the
# exact scaler + KMeans model fit in Section 6e -- nothing is refit here, so train-time
# and inference-time cluster assignments are guaranteed to be produced the same way.
if "applicant_cluster" in FINAL_FEATURES:
    print(f"applicant_cluster is in FINAL_FEATURES -- inference must reuse "
          f"`applicant_scaler` / `applicant_kmeans` fit on {APPLICANT_CLUSTER_FEATURES} in Section 6e.")
if "program_cluster" in FINAL_FEATURES:
    print(f"program_cluster is in FINAL_FEATURES -- inference must reuse "
          f"`university_cluster_scaler` / `university_cluster_model` fit on {UNIV_CLUSTER_COLS} in Section 6e.")

# Historical note: an earlier version of this section tried an additional applicant-strength
# composite signal (built from gpa_gap/gre_gap, standing in for richer per-applicant data
# Agent 1 doesn't have yet) ALONGSIDE the mandatory tier_score, to see whether it added
# anything on top of tier_score + engineered features. It measured no reliable lift and is
# not part of the production feature set. Importantly, it was never used to decide whether
# to keep tier_score itself -- tier_score is never optional -- so it's omitted here rather
# than kept in a form that could be misread as an "Agent 1 tier on/off" test, which this
# architecture explicitly forbids.


Agent 1 Tier representation locked in: {'column': 'tier_score', 'dtype': 'float', 'valid_range': (59.79, 91.5), 'source': 'agent2_training.csv (pre-merged with agent2_enhanced_program_catalog.csv)'}
applicant_cluster is in FINAL_FEATURES -- inference must reuse `applicant_scaler` / `applicant_kmeans` fit on ['gpa_numeric', 'gre_numeric'] in Section 6e.


In [6]:
# Fresh model and calibrated ensemble: no validation or test rows in fitting.
m.loc[m['gre_missing'].eq(1), 'gre_numeric'] = 0.0
Xtr, ytr = m.loc[fit_idx, FINAL_FEATURES], y.loc[fit_idx]
Xval, yval = m.loc[val_idx, FINAL_FEATURES], y.loc[val_idx]
Xte, yte = m.loc[test_idx, FINAL_FEATURES], y.loc[test_idx]
assert set(fit_idx).isdisjoint(val_idx) and set(fit_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx)
tuned_model = GradientBoostingClassifier(**PRODUCTION_PARAMS).fit(Xtr, ytr)
admit_model = CalibratedClassifierCV(tuned_model, method='sigmoid', cv=5)
admit_model.fit(Xtr, ytr)
print('Fit/validation/test rows:', len(fit_idx), len(val_idx), len(test_idx))
print('Fitted:', type(admit_model).__name__, 'with', admit_model.n_features_in_, 'features')


Fit/validation/test rows: 10232 2558 3198
Fitted: CalibratedClassifierCV with 5 features


In [7]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, brier_score_loss, confusion_matrix
positive = list(admit_model.classes_).index(1)
validation_probability = admit_model.predict_proba(Xval)[:, positive]
best_threshold, best_val_f1 = max(((float(t), float(f1_score(yval, validation_probability >= t)))
    for t in np.arange(.05, .96, .01)), key=lambda pair: pair[1])
probability = admit_model.predict_proba(Xte)[:, positive]
prediction = probability >= best_threshold
metrics = {'accuracy': float(accuracy_score(yte, prediction)), 'f1': float(f1_score(yte, prediction)),
    'precision': float(precision_score(yte, prediction)), 'recall': float(recall_score(yte, prediction)),
    'roc_auc': float(roc_auc_score(yte, probability)), 'brier': float(brier_score_loss(yte, probability)),
    'threshold': best_threshold, 'confusion_matrix': confusion_matrix(yte, prediction).tolist(),
    'fit_rows': len(fit_idx), 'validation_rows': len(val_idx), 'test_rows': len(test_idx),
    'scope': 'Row-holdout metrics on supplied dataset; not validated real admissions accuracy.'}
(REPORT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
(REPORT_DIR / 'split_indices.json').write_text(json.dumps({'fit': list(map(int, fit_idx)),
    'validation': list(map(int, val_idx)), 'test': list(map(int, test_idx))}), encoding='utf-8')
print(json.dumps(metrics, indent=2))


{
  "accuracy": 0.8727329580988118,
  "f1": 0.7988136431043006,
  "precision": 0.8568398727465536,
  "recall": 0.7481481481481481,
  "roc_auc": 0.9338590301822124,
  "brier": 0.08445398236630776,
  "threshold": 0.42000000000000004,
  "confusion_matrix": [
    [
      1983,
      135
    ],
    [
      272,
      808
    ]
  ],
  "fit_rows": 10232,
  "validation_rows": 2558,
  "test_rows": 3198,
  "scope": "Row-holdout metrics on supplied dataset; not validated real admissions accuracy."
}


In [8]:
import shap

# Model-agnostic: wraps predict_proba directly, so this works whether tuned_model ended
# up being linear (LogisticRegression) or tree-based -- no assumption about model family.
background = shap.sample(Xtr, 100, random_state=42)
shap_explainer = shap.Explainer(tuned_model.predict_proba, background)

sample_shap = shap_explainer(Xte.iloc[[0]])
print("SHAP values shape:", sample_shap.values.shape)

POSITIVE_CLASS_INDEX = list(tuned_model.classes_).index(1) if hasattr(tuned_model, "classes_") else 1
print("Positive class (admit=1) at index:", POSITIVE_CLASS_INDEX)


def get_shap_top_factors(X_row_df, top_n=3):
    values = shap_explainer(X_row_df)
    vals = values.values[0][:, POSITIVE_CLASS_INDEX] if values.values.ndim == 3 else values.values[0]
    pairs = sorted(zip(X_row_df.columns, vals), key=lambda p: abs(p[1]), reverse=True)
    return [{"feature": f, "impact": round(float(v), 4)} for f, v in pairs[:top_n]]


print("\nTop factors for one test row:", get_shap_top_factors(Xte.iloc[[0]]))

SHAP values shape: (1, 5, 2)
Positive class (admit=1) at index: 1

Top factors for one test row: [{'feature': 'gre_numeric', 'impact': -0.0888}, {'feature': 'gpa_numeric', 'impact': 0.0436}, {'feature': 'tier_score', 'impact': 0.0129}]


C:\Users\ASUS\Downloads\Meridian\Major_Project-2027-main\.venv-backend\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
print("admit_model (calibrated):", type(admit_model))
print("tuned_model (uncalibrated, used for SHAP):", type(tuned_model))
print("shap_explainer:", type(shap_explainer))
print("university_cluster_model:", type(university_cluster_model))

admit_model (calibrated): <class 'sklearn.calibration.CalibratedClassifierCV'>
tuned_model (uncalibrated, used for SHAP): <class 'sklearn.ensemble._gb.GradientBoostingClassifier'>
shap_explainer: <class 'shap.explainers._exact.ExactExplainer'>
university_cluster_model: <class 'sklearn.cluster._kmeans.KMeans'>


In [10]:
import joblib
joblib.dump(admit_model, EXPORT_DIR / 'agent2_admission_model.joblib')
joblib.dump(FINAL_FEATURES, EXPORT_DIR / 'agent2_feature_columns.joblib')
joblib.dump(AGENT1_TIER_FEATURES, EXPORT_DIR / 'agent2_agent1_tier_features.joblib')
joblib.dump(AGENT1_TIER_REPRESENTATION, EXPORT_DIR / 'agent2_agent1_tier_representation.joblib')
joblib.dump(best_threshold, EXPORT_DIR / 'agent2_decision_threshold.joblib')
joblib.dump(university_cluster_model, EXPORT_DIR / 'university_cluster_model.joblib')
joblib.dump(university_cluster_scaler, EXPORT_DIR / 'university_cluster_scaler.joblib')
joblib.dump(UNIV_CLUSTER_COLS, EXPORT_DIR / 'university_cluster_features.joblib')
joblib.dump(univ_cluster_medians, EXPORT_DIR / 'university_cluster_medians.joblib')
joblib.dump(applicant_kmeans, EXPORT_DIR / 'applicant_cluster_model.joblib')
joblib.dump(applicant_scaler, EXPORT_DIR / 'applicant_cluster_scaler.joblib')
joblib.dump(applicant_cluster_medians, EXPORT_DIR / 'applicant_cluster_medians.joblib')
joblib.dump(APPLICANT_CLUSTER_FEATURES, EXPORT_DIR / 'applicant_cluster_features.joblib')
joblib.dump(shap_explainer, EXPORT_DIR / 'agent2_shap_explainer.joblib')

['C:\\Users\\ASUS\\Downloads\\Meridian\\Major_Project-2027-main\\backend\\app\\agents\\university_matching_agent\\trained_models_local\\agent2_shap_explainer.joblib']

In [11]:
# Manifest ties the exported objects, schema, data and runtime to this run.
from meridian.services.agent2 import ARTIFACT_NAMES
versions = {name: importlib.metadata.version(name) for name in
    ['scikit-learn', 'numpy', 'pandas', 'scipy', 'joblib', 'shap', 'numba', 'llvmlite', 'cloudpickle']}
manifest = {'schema_version': 1, 'bundle_id': 'agent2-local-20261012',
    'features': FINAL_FEATURES, 'parameters': PRODUCTION_PARAMS, 'versions': versions,
    'artifacts': {name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest() for name in ARTIFACT_NAMES},
    'source_notebook_sha256': hashlib.sha256((ROOT / 'backend/app/agents/university_matching_agent/Agent2_University_Agent.ipynb').read_bytes()).hexdigest(),
    'catalog_sha256': hashlib.sha256((ROOT / 'data/raw/agent2/agent2_enhanced_program_catalog.csv').read_bytes()).hexdigest(),
    'training_data_sha256': hashlib.sha256((ROOT / 'data/raw/agent2/agent2_training.csv').read_bytes()).hexdigest(),
    'evaluation': metrics, 'explanation': 'SHAP class-1 probability of the uncalibrated classifier',
    'limitations': ['Unverified dataset provenance and conflicting repeated applicant identities',
        'No validated unseen-student/admissions accuracy', 'Agent 7 affects matching score only']}
(EXPORT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print('Exported all', len(manifest['artifacts']), 'artifacts from this run')
print('Versions:', versions)


Exported all 14 artifacts from this run
Versions: {'scikit-learn': '1.9.1', 'numpy': '2.5.4', 'pandas': '3.1.0', 'scipy': '1.18.1', 'joblib': '1.6.0', 'shap': '0.54.1', 'numba': '0.68.0', 'llvmlite': '0.50.0', 'cloudpickle': '3.1.2'}
